# Кросс-валидация полученных моделей

### TODO:

- Пробрасывать категориальные переменные в модели для обучения
- Починить обработку CatBoost (потенциально написать враппер)
- Изменить логи для отображения информации
- Обработка пропусков

------------

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm
import time
import pandas as pd

import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import numpy as np
from IPython.display import display, Markdown
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    precision_recall_curve,
    auc,
    confusion_matrix,
    recall_score,
    precision_score,
)
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

In [ ]:
data = pd.read_csv("./data_raw/clean_table.csv", index_col=0)

In [ ]:
scorers = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, pos_label=1),
    "bal_acc": make_scorer(balanced_accuracy_score),
}

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

In [ ]:
data["Смерть"] = data["Смерть"].map({"Да": 1, "Нет": 0})
data.dropna(subset=["Смерть"], inplace=True)

X = data.drop(["Смерть"], axis=1)
y = data["Смерть"]


def clean_column_names(df):
    """
    Очищает имена колонок от символов, которые не принимает XGBoost
    """
    new_columns = []
    for col in df.columns:
        # Удаляем запрещенные символы
        clean_col = col.replace("[", "_").replace("]", "_").replace("<", "_lt_")
        # Удаляем другие потенциально проблемные символы
        clean_col = clean_col.replace(">", "_gt_").replace(" ", "_").replace(":", "_")
        clean_col = clean_col.replace("(", "_").replace(")", "_").replace(",", "_")
        clean_col = clean_col.replace("{", "_").replace("}", "_").replace("|", "_")
        # Удаляем последовательные подчеркивания
        clean_col = "_".join(filter(None, clean_col.split("_")))
        # Удаляем подчеркивания в начале и конце
        clean_col = clean_col.strip("_")
        # Если имя начинается с цифры, добавляем префикс
        if clean_col and clean_col[0].isdigit():
            clean_col = "f_" + clean_col
        new_columns.append(clean_col)

    # Создаем копию DataFrame с новыми именами колонок
    df_clean = df.copy()
    df_clean.columns = new_columns
    return df_clean


# Применяем к вашим данным
X_clean = clean_column_names(X)

object_cols = X_clean.select_dtypes(include=["object"]).columns

for col in object_cols:
    le = LabelEncoder()
    # Обучаем encoder на объединённых данных, чтобы избежать ошибок с новыми категориями в тесте
    combined = pd.concat([X_clean[col], X_clean[col]], axis=0).astype(str)
    le.fit(combined)
    X_clean[col] = le.transform(X_clean[col].astype(str))

X_clean.drop(["Код_пациента"], axis=1, inplace=True)

In [ ]:
scale_pos_weight = len(y[y == 0]) / max(1, len(y[y == 1]))
class_weight_dict = {0: 1, 1: scale_pos_weight}

In [ ]:
models = {
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        # missing=np.nan,  # явно указываем, что NaN - это пропуски
        random_state=42,
        eval_metric="logloss",  # для бинарной классификации
        tree_method="auto",  # автоматический выбор метода обучения
        enable_categorical=True,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        class_weight=class_weight_dict,  # борьба с дисбалансом
        random_state=42,
        verbose=-1,  # подавляем вывод (задается при инициализации, а не в fit)
    ),
    "CatBoost": CatBoostClassifier(
        iterations=100,
        depth=5,
        learning_rate=0.1,
        class_weights=[1, scale_pos_weight],  # борьба с дисбалансом
        random_state=42,
        verbose=False,
        eval_metric="Logloss",
    ),
}

In [ ]:
results = {}
for name, model in models.items():
    print(f"\n🔄 CV для {name}...")
    cv_res = cross_validate(model, X_clean, y, cv=cv, scoring=scorers, n_jobs=-1, return_train_score=False)
    results[name] = cv_res


🔄 CV для XGBoost...

🔄 CV для LightGBM...

🔄 CV для CatBoost...


AttributeError: The following error was raised: 'CatBoostClassifier' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.

In [ ]:
print("\n" + "=" * 80)
print("📈 Честные метрики (Repeated Stratified 5-Fold CV, 10 повторов = 50 оценок)")
print("=" * 80)
print(f"{'Model':<12} | {'PR-AUC':<18} | {'ROC-AUC':<18} | {'F1':<18} | {'Bal Acc':<18}")
print("-" * 80)


def fmt_ci(scores):
    m = np.mean(scores)
    ci_l, ci_h = np.percentile(scores, [2.5, 97.5])
    return f"{m:.3f} [{ci_l:.3f}–{ci_h:.3f}]"


for name, res in results.items():
    print(
        f"{name:<12} | {fmt_ci(res['test_pr_auc']):<18} | {fmt_ci(res['test_roc_auc']):<18} | "
        f"{fmt_ci(res['test_f1']):<18} | {fmt_ci(res['test_bal_acc']):<18}"
    )


📈 Честные метрики (Repeated Stratified 5-Fold CV, 10 повторов = 50 оценок)
Model        | PR-AUC             | ROC-AUC            | F1                 | Bal Acc           
--------------------------------------------------------------------------------
XGBoost      | 0.700 [0.617–0.757] | 0.958 [0.934–0.972] | 0.593 [0.547–0.642] | 0.858 [0.818–0.899]
LightGBM     | 0.702 [0.631–0.760] | 0.958 [0.937–0.973] | 0.595 [0.547–0.640] | 0.858 [0.818–0.896]


-------------

### Another CV

In [ ]:
X_cv = X_clean.drop(columns=["Name", "Код пациента"], errors="ignore").copy()
y_cv = y.copy()

In [ ]:
groups = X_clean["Name"].astype(str).str.strip().str.upper()

In [ ]:
groups

47         533
106       9476
158        384
568       5568
687       7642
         ...  
6503     13532
6505     10130
6506      6030
6485     10534
17429     9159
Name: Name, Length: 16613, dtype: object

In [ ]:
scorers = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, pos_label=1),
    "bal_acc": make_scorer(balanced_accuracy_score),
}

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
models = {
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        # missing=np.nan,
        random_state=42,
        eval_metric="logloss",
        tree_method="auto",
        enable_categorical=True,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, class_weight=class_weight_dict, random_state=42, verbose=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=100,
        depth=5,
        learning_rate=0.1,
        class_weights=[1, scale_pos_weight],
        random_state=42,
        verbose=False,
        eval_metric="Logloss",
    ),
}

In [ ]:
results = {}
for name, model in models.items():
    print(f"\n🔄 StratifiedGroupCV для {name}...")
    cv_res = cross_validate(
        model, X_cv, y_cv, cv=sgkf, groups=groups, scoring=scorers, n_jobs=-1, return_train_score=False
    )
    results[name] = cv_res


🔄 StratifiedGroupCV для XGBoost...

🔄 StratifiedGroupCV для LightGBM...

🔄 StratifiedGroupCV для CatBoost...


AttributeError: The following error was raised: 'CatBoostClassifier' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.

In [ ]:
print("\n" + "=" * 80)
print("📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)")
print("=" * 80)
print(f"{'Model':<12} | {'PR-AUC':<18} | {'ROC-AUC':<18} | {'F1':<18} | {'Bal Acc':<18}")
print("-" * 80)


def fmt_ci(scores):
    m = np.mean(scores)
    ci_l, ci_h = np.percentile(scores, [2.5, 97.5])
    return f"{m:.3f} [{ci_l:.3f}–{ci_h:.3f}]"


for mdl, res in results.items():
    print(
        f"{mdl:<12} | {fmt_ci(res['test_pr_auc']):<18} | {fmt_ci(res['test_roc_auc']):<18} | "
        f"{fmt_ci(res['test_f1']):<18} | {fmt_ci(res['test_bal_acc']):<18}"
    )


📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)
Model        | PR-AUC             | ROC-AUC            | F1                 | Bal Acc           
--------------------------------------------------------------------------------
XGBoost      | 0.690 [0.650–0.720] | 0.955 [0.947–0.965] | 0.578 [0.545–0.616] | 0.847 [0.823–0.869]
LightGBM     | 0.690 [0.642–0.717] | 0.954 [0.944–0.965] | 0.586 [0.539–0.626] | 0.858 [0.824–0.871]


In [ ]:
sum(y), len(X_clean) - sum(y)

(575.0, 16038.0)

------------

### With hyperparameters

In [ ]:
models = {
    "XGBoost": XGBClassifier(
        n_estimators=100,
        min_child_weight=15,
        subsample=0.8745297951606058,
        colsample_bytree=0.4954110056842209,
        gamma=5.233186685488018,
        reg_alpha=0.0005757915861725465,
        reg_lambda=4.198614550306633,
        max_depth=74,
        learning_rate=0.3705492544881076,
        scale_pos_weight=24.40033721777531,
        # missing=np.nan,
        random_state=42,
        eval_metric="logloss",
        tree_method="auto",
        enable_categorical=True,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        num_leaves=220,
        learning_rate=0.30906593527878456,
        min_child_samples=69,
        subsample=0.8701085058859857,
        colsample_bytree=0.4868258750883674,
        reg_alpha=0.023127138648795138,
        reg_lambda=0.06691267437317566,
        min_split_gain=0.1116982770332515,
        max_depth=8,
        class_weight=class_weight_dict,
        random_state=42,
        verbose=-1,
    ),
}

In [ ]:
results = {}
for name, model in models.items():
    print(f"\n🔄 StratifiedGroupCV для {name}...")
    cv_res = cross_validate(
        model, X_cv, y_cv, cv=sgkf, groups=groups, scoring=scorers, n_jobs=-1, return_train_score=False
    )
    results[name] = cv_res


🔄 StratifiedGroupCV для XGBoost...

🔄 StratifiedGroupCV для LightGBM...


In [ ]:
# 6. Вывод с 95% ДИ
print("\n" + "=" * 80)
print("📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)")
print("=" * 80)
print(f"{'Model':<12} | {'PR-AUC':<18} | {'ROC-AUC':<18} | {'F1':<18} | {'Bal Acc':<18}")
print("-" * 80)


def fmt_ci(scores):
    m = np.mean(scores)
    ci_l, ci_h = np.percentile(scores, [2.5, 97.5])
    return f"{m:.3f} [{ci_l:.3f}–{ci_h:.3f}]"


for mdl, res in results.items():
    print(
        f"{mdl:<12} | {fmt_ci(res['test_pr_auc']):<18} | {fmt_ci(res['test_roc_auc']):<18} | "
        f"{fmt_ci(res['test_f1']):<18} | {fmt_ci(res['test_bal_acc']):<18}"
    )


📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)
Model        | PR-AUC             | ROC-AUC            | F1                 | Bal Acc           
--------------------------------------------------------------------------------
XGBoost      | 0.689 [0.670–0.716] | 0.957 [0.951–0.963] | 0.615 [0.574–0.643] | 0.830 [0.790–0.847]
LightGBM     | 0.716 [0.683–0.752] | 0.955 [0.941–0.966] | 0.654 [0.608–0.684] | 0.802 [0.775–0.820]


---------

### Stemi YES Hyperparameters NO

In [ ]:
data = data[data["STEMI"] == "Да"]

In [ ]:
data.drop("STEMI", axis=1, inplace=True)

In [ ]:
data["Смерть"] = data["Смерть"].map({"Да": 1, "Нет": 0})
data.dropna(subset=["Смерть"], inplace=True)

X = data.drop(["Смерть"], axis=1)
y = data["Смерть"]


def clean_column_names(df):
    """
    Очищает имена колонок от символов, которые не принимает XGBoost
    """
    new_columns = []
    for col in df.columns:
        # Удаляем запрещенные символы
        clean_col = col.replace("[", "_").replace("]", "_").replace("<", "_lt_")
        # Удаляем другие потенциально проблемные символы
        clean_col = clean_col.replace(">", "_gt_").replace(" ", "_").replace(":", "_")
        clean_col = clean_col.replace("(", "_").replace(")", "_").replace(",", "_")
        clean_col = clean_col.replace("{", "_").replace("}", "_").replace("|", "_")
        # Удаляем последовательные подчеркивания
        clean_col = "_".join(filter(None, clean_col.split("_")))
        # Удаляем подчеркивания в начале и конце
        clean_col = clean_col.strip("_")
        # Если имя начинается с цифры, добавляем префикс
        if clean_col and clean_col[0].isdigit():
            clean_col = "f_" + clean_col
        new_columns.append(clean_col)

    # Создаем копию DataFrame с новыми именами колонок
    df_clean = df.copy()
    df_clean.columns = new_columns
    return df_clean


# Применяем к вашим данным
X_clean = clean_column_names(X)

object_cols = X_clean.select_dtypes(include=["object"]).columns

for col in object_cols:
    le = LabelEncoder()
    combined = pd.concat([X_clean[col], X_clean[col]], axis=0).astype(str)
    le.fit(combined)
    X_clean[col] = le.transform(X_clean[col].astype(str))

X_clean.drop(["Код_пациента"], axis=1, inplace=True)

In [ ]:
X_cv = X_clean.drop(columns=["Name", "Код пациента"], errors="ignore").copy()
y_cv = y.copy()

In [ ]:
groups = X_clean["Name"].astype(str).str.strip().str.upper()

In [ ]:
scorers = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, pos_label=1),
    "bal_acc": make_scorer(balanced_accuracy_score),
}

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
models = {
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,  # борьба с дисбалансом
        # missing=np.nan,  # явно указываем, что NaN - это пропуски
        random_state=42,
        eval_metric="logloss",  # для бинарной классификации
        tree_method="auto",  # автоматический выбор метода обучения
        enable_categorical=True,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        class_weight=class_weight_dict,  # борьба с дисбалансом
        random_state=42,
        verbose=-1,  # подавляем вывод (задается при инициализации, а не в fit)
    ),
    "CatBoost": CatBoostClassifier(
        iterations=100,
        depth=5,
        learning_rate=0.1,
        class_weights=[1, scale_pos_weight],  # борьба с дисбалансом
        random_state=42,
        verbose=False,
        eval_metric="Logloss",
    ),
}

In [ ]:
results = {}
for name, model in models.items():
    print(f"\n🔄 StratifiedGroupCV для {name}...")
    cv_res = cross_validate(
        model,
        X_cv,
        y_cv,
        cv=sgkf,
        groups=groups,  # ← Ключевой параметр
        scoring=scorers,
        n_jobs=-1,
        return_train_score=False,
    )
    results[name] = cv_res


🔄 StratifiedGroupCV для XGBoost...

🔄 StratifiedGroupCV для LightGBM...

🔄 StratifiedGroupCV для CatBoost...


AttributeError: The following error was raised: 'CatBoostClassifier' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.

In [ ]:
# 6. Вывод с 95% ДИ
print("\n" + "=" * 80)
print("📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)")
print("=" * 80)
print(f"{'Model':<12} | {'PR-AUC':<18} | {'ROC-AUC':<18} | {'F1':<18} | {'Bal Acc':<18}")
print("-" * 80)


def fmt_ci(scores):
    m = np.mean(scores)
    ci_l, ci_h = np.percentile(scores, [2.5, 97.5])
    return f"{m:.3f} [{ci_l:.3f}–{ci_h:.3f}]"


for mdl, res in results.items():
    print(
        f"{mdl:<12} | {fmt_ci(res['test_pr_auc']):<18} | {fmt_ci(res['test_roc_auc']):<18} | "
        f"{fmt_ci(res['test_f1']):<18} | {fmt_ci(res['test_bal_acc']):<18}"
    )


📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)
Model        | PR-AUC             | ROC-AUC            | F1                 | Bal Acc           
--------------------------------------------------------------------------------
XGBoost      | 0.848 [0.807–0.868] | 0.969 [0.962–0.974] | 0.756 [0.710–0.804] | 0.878 [0.861–0.899]
LightGBM     | 0.841 [0.797–0.858] | 0.968 [0.962–0.971] | 0.748 [0.710–0.793] | 0.875 [0.856–0.890]


-----------------

### STEMI YES HYPERPARAMETERS YES

In [ ]:
data = data[data["STEMI"] == "Да"]

In [ ]:
data.drop("STEMI", axis=1, inplace=True)

In [ ]:
data["Смерть"] = data["Смерть"].map({"Да": 1, "Нет": 0})
data.dropna(subset=["Смерть"], inplace=True)

X = data.drop(["Смерть"], axis=1)
y = data["Смерть"]


def clean_column_names(df):
    """
    Очищает имена колонок от символов, которые не принимает XGBoost
    """
    new_columns = []
    for col in df.columns:
        # Удаляем запрещенные символы
        clean_col = col.replace("[", "_").replace("]", "_").replace("<", "_lt_")
        # Удаляем другие потенциально проблемные символы
        clean_col = clean_col.replace(">", "_gt_").replace(" ", "_").replace(":", "_")
        clean_col = clean_col.replace("(", "_").replace(")", "_").replace(",", "_")
        clean_col = clean_col.replace("{", "_").replace("}", "_").replace("|", "_")
        # Удаляем последовательные подчеркивания
        clean_col = "_".join(filter(None, clean_col.split("_")))
        # Удаляем подчеркивания в начале и конце
        clean_col = clean_col.strip("_")
        # Если имя начинается с цифры, добавляем префикс
        if clean_col and clean_col[0].isdigit():
            clean_col = "f_" + clean_col
        new_columns.append(clean_col)

    # Создаем копию DataFrame с новыми именами колонок
    df_clean = df.copy()
    df_clean.columns = new_columns
    return df_clean


# Применяем к вашим данным
X_clean = clean_column_names(X)

object_cols = X_clean.select_dtypes(include=["object"]).columns

for col in object_cols:
    le = LabelEncoder()
    # Обучаем encoder на объединённых данных, чтобы избежать ошибок с новыми категориями в тесте
    combined = pd.concat([X_clean[col], X_clean[col]], axis=0).astype(str)
    le.fit(combined)
    X_clean[col] = le.transform(X_clean[col].astype(str))

X_clean.drop(["Код_пациента"], axis=1, inplace=True)

In [ ]:
X_cv = X_clean.drop(columns=["Name", "Код пациента"], errors="ignore").copy()
y_cv = y.copy()

In [ ]:
groups = X_clean["Name"].astype(str).str.strip().str.upper()

In [ ]:
scorers = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, pos_label=1),
    "bal_acc": make_scorer(balanced_accuracy_score),
}

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
models = {
    "XGBoost": XGBClassifier(
        n_estimators=100,
        min_child_weight=15,
        subsample=0.8745297951606058,
        colsample_bytree=0.4954110056842209,
        gamma=5.233186685488018,
        reg_alpha=0.0005757915861725465,
        reg_lambda=4.198614550306633,
        max_depth=74,
        learning_rate=0.3705492544881076,
        scale_pos_weight=24.40033721777531,
        missing=np.nan,
        random_state=42,
        eval_metric="logloss",
        tree_method="auto",
        enable_categorical=True,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        num_leaves=220,
        learning_rate=0.30906593527878456,
        min_child_samples=69,
        subsample=0.8701085058859857,
        colsample_bytree=0.4868258750883674,
        reg_alpha=0.023127138648795138,
        reg_lambda=0.06691267437317566,
        min_split_gain=0.1116982770332515,
        max_depth=8,
        class_weight=class_weight_dict,
        random_state=42,
        verbose=-1,
    ),
}

In [ ]:
results = {}
for name, model in models.items():
    print(f"\n🔄 StratifiedGroupCV для {name}...")
    cv_res = cross_validate(
        model, X_cv, y_cv, cv=sgkf, groups=groups, scoring=scorers, n_jobs=-1, return_train_score=False
    )
    results[name] = cv_res


🔄 StratifiedGroupCV для XGBoost...

🔄 StratifiedGroupCV для LightGBM...


In [ ]:
print("\n" + "=" * 80)
print("📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)")
print("=" * 80)
print(f"{'Model':<12} | {'PR-AUC':<18} | {'ROC-AUC':<18} | {'F1':<18} | {'Bal Acc':<18}")
print("-" * 80)


def fmt_ci(scores):
    m = np.mean(scores)
    ci_l, ci_h = np.percentile(scores, [2.5, 97.5])
    return f"{m:.3f} [{ci_l:.3f}–{ci_h:.3f}]"


for mdl, res in results.items():
    print(
        f"{mdl:<12} | {fmt_ci(res['test_pr_auc']):<18} | {fmt_ci(res['test_roc_auc']):<18} | "
        f"{fmt_ci(res['test_f1']):<18} | {fmt_ci(res['test_bal_acc']):<18}"
    )


📈 Метрики (StratifiedGroupKFold, группировка по Name, 5 фолдов)
Model        | PR-AUC             | ROC-AUC            | F1                 | Bal Acc           
--------------------------------------------------------------------------------
XGBoost      | 0.811 [0.767–0.829] | 0.961 [0.955–0.966] | 0.714 [0.668–0.739] | 0.865 [0.834–0.890]
LightGBM     | 0.839 [0.813–0.851] | 0.966 [0.959–0.970] | 0.764 [0.738–0.791] | 0.849 [0.837–0.867]
